# XGBoost Regression for Exam Score Prediction

This notebook trains an XGBoost regressor to predict a student's exam score using the available study and academic features.
Each code cell includes comments to explain the purpose of the step and the logic used.

In [1]:
# Step 1: Import the modules required for data loading, preprocessing, and modeling.
from pathlib import Path

import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from xgboost import XGBRegressor

RANDOM_STATE = 42


In [2]:
# Step 2: Load the dataset from the repository and inspect the first rows.
project_root = Path.cwd()
data_candidates = [
    project_root / 'data loading' / 'Exam_Score_Prediction.csv',
    project_root.parent / 'data loading' / 'Exam_Score_Prediction.csv',
]
data_path = next((candidate for candidate in data_candidates if candidate.exists()), data_candidates[0])
df = pd.read_csv(data_path)

print(f'Dataset shape: {df.shape}')
df.head()


Dataset shape: (20000, 13)


,student_id,age,gender,course,study_hours,class_attendance,internet_access,sleep_hours,sleep_quality,study_method,facility_rating,exam_difficulty,exam_score
0,1,17,male,diploma,2.78,92.9,yes,7.4,poor,coaching,low,hard,58.9
1,2,23,other,bca,3.37,64.8,yes,4.6,average,online videos,medium,moderate,54.8
2,3,22,male,b.sc,7.88,76.8,yes,8.5,poor,coaching,high,moderate,90.3
3,4,20,other,diploma,0.67,48.4,yes,5.8,average,online videos,low,moderate,29.7
4,5,20,female,diploma,0.89,71.6,yes,9.8,poor,coaching,low,moderate,43.7


In [3]:
# Step 3: Check for missing values to confirm the data is ready for modeling.
missing_values = df.isna().sum()
print(missing_values[missing_values > 0])

# If there are no missing values, this confirms we can proceed without extra cleanup.
df.info()


Series([], dtype: int64)
<class 'pandas.DataFrame'>
RangeIndex: 20000 entries, 0 to 19999
Data columns (total 13 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   student_id        20000 non-null  int64  
 1   age               20000 non-null  int64  
 2   gender            20000 non-null  str    
 3   course            20000 non-null  str    
 4   study_hours       20000 non-null  float64
 5   class_attendance  20000 non-null  float64
 6   internet_access   20000 non-null  str    
 7   sleep_hours       20000 non-null  float64
 8   sleep_quality     20000 non-null  str    
 9   study_method      20000 non-null  str    
 10  facility_rating   20000 non-null  str    
 11  exam_difficulty   20000 non-null  str    
 12  exam_score        20000 non-null  float64
dtypes: float64(4), int64(2), str(7)
memory usage: 2.7 MB


## Feature and target selection

We separate the target variable, `exam_score`, from the input features. Categorical features are encoded automatically, while numerical features are handled with median imputation.

In [33]:
# Step 4: Define the target and drop the non-feature columns from the input matrix.
target_col = 'exam_score'
features = df.drop(columns=['student_id','study_method', 'sleep_quality',target_col])
target = df[target_col]

# Show the first few feature names so the model input is clearly understood.
print('Feature columns:', list(features.columns))
print(f'Target column: {target_col}')


Feature columns: ['age', 'gender', 'course', 'study_hours', 'class_attendance', 'internet_access', 'sleep_hours', 'facility_rating', 'exam_difficulty']
Target column: exam_score


In [34]:
# Step 5: Split the data into training and test sets for honest evaluation.
X_train, X_test, y_train, y_test = train_test_split(
    features,
    target,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

print(f'Training rows: {len(X_train)}')
print(f'Test rows: {len(X_test)}')


Training rows: 16000
Test rows: 4000


In [35]:
# Step 6: Create separate preprocessing pipelines for numeric and categorical fields.
numeric_features = features.select_dtypes(exclude=['object']).columns.tolist()
categorical_features = features.select_dtypes(include=['object']).columns.tolist()

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median'))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore'))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('numeric', numeric_transformer, numeric_features),
        ('categorical', categorical_transformer, categorical_features),
    ]
)

print('Numeric columns:', numeric_features)
print('Categorical columns:', categorical_features)


Numeric columns: ['age', 'study_hours', 'class_attendance', 'sleep_hours']
Categorical columns: ['gender', 'course', 'internet_access', 'facility_rating', 'exam_difficulty']


C:\Users\Malik\AppData\Local\Temp\ipykernel_29624\3094570685.py:3: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  categorical_features = features.select_dtypes(include=['object']).columns.tolist()


In [36]:
# Step 7: Build the XGBoost regressor and wrap it in a pipeline with the preprocessor.
model = XGBRegressor(
    objective='reg:squarederror',
    n_estimators=500,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=RANDOM_STATE
)

pipeline = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        ('model', model),
    ]
)



from datetime import  datetime



print (datetime.now())
pipeline.fit(X_train, y_train)


print (datetime.now())
print('Model training complete.')


#0.08ms

2026-11-06 21:22:32.167851
2026-11-06 21:22:32.818666
Model training complete.


In [37]:
fitted_preprocessor = pipeline.named_steps["preprocessor"]
matrix = fitted_preprocessor.transform(X_train)

# Convert sparse output to a regular array for display
if hasattr(matrix, "toarray"):
    matrix = matrix.toarray()

feature_matrix = pd.DataFrame(
    matrix,
    columns=fitted_preprocessor.get_feature_names_out(),
    index=X_train.index,
)

print("Matrix shape:", feature_matrix.shape)
display(feature_matrix.head())

Matrix shape: (16000, 22)


,numeric__age,numeric__study_hours,numeric__class_attendance,numeric__sleep_hours,categorical__gender_female,categorical__gender_male,categorical__gender_other,categorical__course_b.com,categorical__course_b.sc,categorical__course_b.tech,...,categorical__course_bca,categorical__course_diploma,categorical__internet_access_no,categorical__internet_access_yes,categorical__facility_rating_high,categorical__facility_rating_low,categorical__facility_rating_medium,categorical__exam_difficulty_easy,categorical__exam_difficulty_hard,categorical__exam_difficulty_moderate
5894,21.0,6.87,57.1,9.0,0.0,0.0,1.0,1.0,0.0,0.0,...,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
3728,19.0,6.91,54.8,9.1,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,1.0,0.0
8958,20.0,5.45,99.4,7.6,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,1.0
7671,20.0,4.73,47.6,7.0,0.0,0.0,1.0,0.0,0.0,0.0,...,0.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0
5999,17.0,3.81,78.6,7.8,1.0,0.0,0.0,0.0,0.0,0.0,...,1.0,0.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,1.0


In [38]:
# Step 8: Evaluate the model on the unseen test set using standard regression metrics.
predictions = pipeline.predict(X_test)

mae = mean_absolute_error(y_test, predictions)
mse = mean_squared_error(y_test, predictions)
rmse = mse ** 0.5
r2 = r2_score(y_test, predictions)

print(f'MAE: {mae:.4f}')
print(f'MSE: {mse:.4f}')
print(f'RMSE: {rmse:.4f}')
print(f'R^2 Score: {r2:.4f}')


MAE: 9.2585
MSE: 131.5216
RMSE: 11.4683
R^2 Score: 0.6323


In [39]:
# Step 9: Inspect a few sample predictions to confirm that the model produces realistic values.
sample = X_test.head(5)
sample_predictions = pipeline.predict(sample)

for idx, value in enumerate(sample_predictions):
    actual = y_test.iloc[idx]
    print(f'Row {idx}: predicted = {value:.2f}, actual = {actual:.2f}')


Row 0: predicted = 41.16, actual = 31.10
Row 1: predicted = 67.69, actual = 81.60
Row 2: predicted = 59.59, actual = 68.00
Row 3: predicted = 93.10, actual = 100.00
Row 4: predicted = 82.92, actual = 84.80


## Final note

The XGBoost regressor is a strong choice for tabular data because it handles non-linear relationships well and uses tree-based boosting to improve prediction quality. The metrics above help determine whether the model is accurate enough for the task.

## Manual testing

Use this section to provide a new student profile manually and predict the exam score with the trained XGBoost model.

In [40]:
# Step 10: Test the trained model with a manually entered student record.
manual_student = {
    'age': 22,
    'gender': 'male',
    'course': 'b.tech',
    'study_hours': 6.5,
    'class_attendance': 88.0,
    'internet_access': 'yes',
    'sleep_hours': 7.5,
    'facility_rating': 'high',
    'exam_difficulty': 'moderate',
}

manual_df = pd.DataFrame([manual_student])
predicted_score = pipeline.predict(manual_df)[0]

print(f'Manual test input: {manual_student}')
print(f'Predicted exam score: {predicted_score:.2f}')


Manual test input: {'age': 22, 'gender': 'male', 'course': 'b.tech', 'study_hours': 6.5, 'class_attendance': 88.0, 'internet_access': 'yes', 'sleep_hours': 7.5, 'facility_rating': 'high', 'exam_difficulty': 'moderate'}
Predicted exam score: 86.09
